# Model Review Notebook

This notebook reviews all saved regression models for the thesis project on predicting cost overruns and schedule overruns. It loads saved `.joblib` models, reconstructs or loads the held-out test data, compares model performance, evaluates early-warning risk detection, and exports thesis-ready tables and figures.

Minimal expected project structure:

```text
project_root/
  data/
    processed/
  outputs/
    models/
    tables/
    figures/
  notebooks/
    model_review.ipynb
```

## Imports

In [ ]:
import os
import sys
import glob
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print(f"SHAP is not available. SHAP plots will be skipped. Reason: {exc}")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (8, 6)

## Paths / configuration

In [ ]:
# Adjust these paths if your project layout changes.
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

MODELS_DIR = os.path.join(PROJECT_ROOT, "outputs", "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
CONFIG_PATH = os.path.join(TABLES_DIR, "run_config.json")

# Explicit test-file assumptions. If these files do not exist, the notebook
# reconstructs the held-out test split from the saved final synthetic dataset.
X_TEST_PATH = os.path.join(DATA_DIR, "processed", "X_test.csv")
Y_COST_TEST_PATH = os.path.join(DATA_DIR, "processed", "y_cost_test.csv")
Y_SCHEDULE_TEST_PATH = os.path.join(DATA_DIR, "processed", "y_schedule_test.csv")
FINAL_DATASET_PATH = os.path.join(DATA_DIR, "processed", "final_synthetic_dataset.csv")

TARGET_COST = "cost_overrun_pct"
TARGET_SCHEDULE = "schedule_overrun_pct"
RISK_BAND_EDGES = (0.05, 0.15)
HIGH_RISK_BINARY_THRESHOLD = 0.15

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

run_config = {}
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as handle:
        run_config = json.load(handle)
    print(f"Loaded run configuration from: {CONFIG_PATH}")
else:
    print("Run configuration file not found. Fallback defaults will be used.")

random_state = run_config.get("random_state", 42)
test_size = run_config.get("test_size", 0.2)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"MODELS_DIR = {MODELS_DIR}")
print(f"DATA_DIR = {DATA_DIR}")

## Helper functions

In [ ]:
def safe_mape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)))


def compute_regression_metrics(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": safe_mape(y_true, y_pred),
        "r2": float(r2_score(y_true, y_pred)),
    }


def infer_target_from_name(name):
    lower = name.lower()
    if "cost" in lower:
        return "cost"
    if "schedule" in lower or "delay" in lower:
        return "schedule"
    print(f"Warning: could not infer target from filename '{name}'.")
    return "unknown"


def infer_model_family(name):
    lower = name.lower()
    if "linear" in lower:
        return "linear_regression"
    if "random_forest" in lower or "forest" in lower:
        return "random_forest"
    if "xgboost" in lower or "xgb" in lower:
        return "xgboost"
    return "unknown"


def assign_risk_category(values, low_edge=0.05, high_edge=0.15):
    values = np.asarray(values)
    return np.where(values < low_edge, "low_risk", np.where(values < high_edge, "moderate_risk", "high_risk"))


def compute_high_risk_metrics(y_true, y_pred, threshold=0.15):
    yt = (np.asarray(y_true) >= threshold).astype(int)
    yp = (np.asarray(y_pred) >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision": float(precision_score(yt, yp, zero_division=0)),
        "recall": float(recall_score(yt, yp, zero_division=0)),
        "f1": float(f1_score(yt, yp, zero_division=0)),
        "tp": int(((yt == 1) & (yp == 1)).sum()),
        "fp": int(((yt == 0) & (yp == 1)).sum()),
        "tn": int(((yt == 0) & (yp == 0)).sum()),
        "fn": int(((yt == 1) & (yp == 0)).sum()),
    }


def get_expected_input_columns(model):
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)
    return None


def validate_model_input(model_name, model, X):
    expected = get_expected_input_columns(model)
    if expected is None:
        print(f"Warning: {model_name} does not expose feature_names_in_. Column-name validation is limited.")
        return X, True

    missing = [col for col in expected if col not in X.columns]
    extra = [col for col in X.columns if col not in expected]
    if missing:
        print(f"Warning: {model_name} is missing required features: {missing}")
        return X, False
    if extra:
        print(f"Warning: {model_name} received extra columns. They will be dropped: {extra}")

    X_aligned = X.reindex(columns=expected)
    return X_aligned, True


def get_transformed_feature_names(model, fallback_columns):
    try:
        if hasattr(model, "named_steps") and "preprocessor" in model.named_steps:
            return list(model.named_steps["preprocessor"].get_feature_names_out())
    except Exception as exc:
        print(f"Warning: failed to extract transformed feature names: {exc}")
    return list(fallback_columns)


## Load data

In [ ]:
def reconstruct_test_split_from_final_dataset(dataset_path, random_state=42, test_size=0.2):
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Core data is missing. Could not find final dataset at: {dataset_path}"
        )

    df = pd.read_csv(dataset_path)
    target_cols = [TARGET_COST, TARGET_SCHEDULE]
    if any(col not in df.columns for col in target_cols):
        raise ValueError(f"Final dataset must contain target columns: {target_cols}")

    X_full = df.drop(columns=target_cols)
    y_full = df[target_cols]
    _, X_test_local, _, y_test_local = train_test_split(
        X_full,
        y_full,
        test_size=test_size,
        random_state=random_state,
    )

    return X_test_local.reset_index(drop=True), y_test_local.reset_index(drop=True)


if os.path.exists(X_TEST_PATH) and os.path.exists(Y_COST_TEST_PATH) and os.path.exists(Y_SCHEDULE_TEST_PATH):
    X_test = pd.read_csv(X_TEST_PATH)
    y_cost_test = pd.read_csv(Y_COST_TEST_PATH).squeeze("columns")
    y_schedule_test = pd.read_csv(Y_SCHEDULE_TEST_PATH).squeeze("columns")
    print("Loaded explicit test files from disk.")
else:
    print("Explicit test files not found. Reconstructing held-out test split from final synthetic dataset.")
    X_test, y_test_df = reconstruct_test_split_from_final_dataset(
        FINAL_DATASET_PATH,
        random_state=random_state,
        test_size=test_size,
    )
    y_cost_test = y_test_df[TARGET_COST]
    y_schedule_test = y_test_df[TARGET_SCHEDULE]

if X_test is None or len(X_test) == 0:
    raise RuntimeError("Core test data could not be loaded or reconstructed.")

print(f"X_test shape: {X_test.shape}")
print(f"Cost test target length: {len(y_cost_test)}")
print(f"Schedule test target length: {len(y_schedule_test)}")
X_test.head()

## Load models

In [ ]:
model_paths = sorted(glob.glob(os.path.join(MODELS_DIR, "*.joblib")))
if not model_paths:
    raise FileNotFoundError(f"No .joblib models were found in: {MODELS_DIR}")

loaded_models = {}
model_registry_rows = []

for path in model_paths:
    name = os.path.splitext(os.path.basename(path))[0]
    try:
        loaded_models[name] = joblib.load(path)
        target_group = infer_target_from_name(name)
        family = infer_model_family(name)
        model_registry_rows.append(
            {
                "model_name": name,
                "path": path,
                "target_group": target_group,
                "model_family": family,
                "loaded": True,
            }
        )
    except Exception as exc:
        print(f"Warning: failed to load model '{path}'. Reason: {exc}")
        model_registry_rows.append(
            {
                "model_name": name,
                "path": path,
                "target_group": infer_target_from_name(name),
                "model_family": infer_model_family(name),
                "loaded": False,
            }
        )

model_registry = pd.DataFrame(model_registry_rows)
model_registry.to_csv(os.path.join(TABLES_DIR, "model_review_loaded_models.csv"), index=False)
model_registry

## Validate feature compatibility and run predictions

In [ ]:
predictions = {"cost": {}, "schedule": {}, "unknown": {}}
prediction_rows = []

for model_name, model in loaded_models.items():
    target_group = infer_target_from_name(model_name)
    X_model, is_valid = validate_model_input(model_name, model, X_test)
    if not is_valid:
        print(f"Skipping {model_name} because input validation failed.")
        continue

    try:
        y_pred = model.predict(X_model)
        predictions[target_group][model_name] = y_pred

        if target_group == "cost":
            y_true = y_cost_test.to_numpy()
        elif target_group == "schedule":
            y_true = y_schedule_test.to_numpy()
        else:
            print(f"Warning: {model_name} target is unknown. Predictions were computed but not evaluated.")
            continue

        prediction_rows.append(
            pd.DataFrame(
                {
                    "model_name": model_name,
                    "target_group": target_group,
                    "actual": y_true,
                    "predicted": y_pred,
                    "residual": y_true - y_pred,
                }
            )
        )
    except Exception as exc:
        print(f"Warning: prediction failed for {model_name}. Reason: {exc}")

if not prediction_rows:
    raise RuntimeError("No models produced valid predictions. Review the warnings above.")

prediction_frame = pd.concat(prediction_rows, ignore_index=True)
prediction_frame.to_csv(os.path.join(TABLES_DIR, "model_review_all_predictions.csv"), index=False)
prediction_frame.head()

## Compute regression metrics and compare models in tables

In [ ]:
metric_rows = []

for target_group in ["cost", "schedule"]:
    y_true = y_cost_test.to_numpy() if target_group == "cost" else y_schedule_test.to_numpy()
    for model_name, y_pred in predictions[target_group].items():
        metrics = compute_regression_metrics(y_true, y_pred)
        metric_rows.append({"target_group": target_group, "model_name": model_name, **metrics})

metrics_df = pd.DataFrame(metric_rows).sort_values(["target_group", "rmse"]).reset_index(drop=True)

cost_metrics_df = metrics_df[metrics_df["target_group"] == "cost"].copy()
schedule_metrics_df = metrics_df[metrics_df["target_group"] == "schedule"].copy()

cost_metrics_path = os.path.join(TABLES_DIR, "model_review_cost_regression_comparison.csv")
schedule_metrics_path = os.path.join(TABLES_DIR, "model_review_schedule_regression_comparison.csv")
cost_metrics_df.to_csv(cost_metrics_path, index=False)
schedule_metrics_df.to_csv(schedule_metrics_path, index=False)

print("Cost model comparison")
display(cost_metrics_df)
print("Schedule model comparison")
display(schedule_metrics_df)

## Plot predicted vs actual and residuals

In [ ]:
for target_group in ["cost", "schedule"]:
    subset = prediction_frame[prediction_frame["target_group"] == target_group].copy()
    for model_name in subset["model_name"].unique():
        model_subset = subset[subset["model_name"] == model_name].copy()
        safe_name = model_name.replace("/", "_")
        try:
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))

            axes[0].scatter(model_subset["actual"], model_subset["predicted"], alpha=0.65)
            xy_min = min(model_subset["actual"].min(), model_subset["predicted"].min())
            xy_max = max(model_subset["actual"].max(), model_subset["predicted"].max())
            axes[0].plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--", color="black")
            axes[0].set_title(f"Predicted vs actual: {model_name}")
            axes[0].set_xlabel("Actual overrun (%)")
            axes[0].set_ylabel("Predicted overrun (%)")

            sns.histplot(model_subset["residual"], kde=True, bins=30, ax=axes[1])
            axes[1].set_title(f"Residual distribution: {model_name}")
            axes[1].set_xlabel("Residual")

            plt.suptitle(f"{target_group.title()} target diagnostics", y=1.02)
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_{target_group}_{safe_name}_diagnostics.png"), dpi=200)
            plt.close()
        except Exception as exc:
            print(f"Warning: could not create diagnostic plots for {model_name}. Reason: {exc}")

## Plot feature importance for tree-based models

In [ ]:
tree_importance_rows = []

for model_name, model in loaded_models.items():
    family = infer_model_family(model_name)
    if family not in {"random_forest", "xgboost"}:
        continue

    try:
        estimator = model.named_steps["model"] if hasattr(model, "named_steps") and "model" in model.named_steps else model
        if not hasattr(estimator, "feature_importances_"):
            print(f"Skipping feature-importance plot for {model_name}: feature_importances_ not available.")
            continue

        feature_names = get_transformed_feature_names(model, X_test.columns)
        importances = estimator.feature_importances_
        imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
        imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)
        imp_df["model_name"] = model_name
        tree_importance_rows.append(imp_df)

        top_df = imp_df.head(15).sort_values("importance", ascending=True)
        plt.figure(figsize=(10, 6))
        sns.barplot(data=top_df, x="importance", y="feature", hue="feature", legend=False, palette="deep")
        plt.title(f"Feature importance: {model_name}")
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, f"model_review_feature_importance_{model_name}.png"), dpi=200)
        plt.close()
    except Exception as exc:
        print(f"Warning: feature-importance plot failed for {model_name}. Reason: {exc}")

tree_importance_df = pd.concat(tree_importance_rows, ignore_index=True) if tree_importance_rows else pd.DataFrame()
if not tree_importance_df.empty:
    tree_importance_df.to_csv(os.path.join(TABLES_DIR, "model_review_tree_feature_importance.csv"), index=False)
    display(tree_importance_df.groupby("model_name", as_index=False).head(10))
else:
    print("No tree-model feature importances were available.")

## SHAP summary plots for final XGBoost models (optional)

In [ ]:
if SHAP_AVAILABLE:
    for target_group in ["cost", "schedule"]:
        xgb_candidates = [name for name in predictions[target_group].keys() if infer_model_family(name) == "xgboost"]
        if not xgb_candidates:
            print(f"No XGBoost model found for target '{target_group}'.")
            continue

        xgb_metrics = metrics_df[
            (metrics_df["target_group"] == target_group)
            & (metrics_df["model_name"].isin(xgb_candidates))
        ].sort_values("rmse")
        best_xgb_name = xgb_metrics.iloc[0]["model_name"]
        best_xgb_model = loaded_models[best_xgb_name]

        try:
            X_aligned, is_valid = validate_model_input(best_xgb_name, best_xgb_model, X_test)
            if not is_valid:
                print(f"Skipping SHAP for {best_xgb_name} because feature validation failed.")
                continue

            preprocessor = best_xgb_model.named_steps["preprocessor"]
            booster = best_xgb_model.named_steps["model"]
            X_transformed = preprocessor.transform(X_aligned)
            feature_names = preprocessor.get_feature_names_out()
            X_shap = X_transformed[:500] if X_transformed.shape[0] > 500 else X_transformed

            explainer = shap.TreeExplainer(booster)
            shap_values = explainer.shap_values(X_shap)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False)
            plt.title(f"SHAP summary: {best_xgb_name}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_shap_summary_{best_xgb_name}.png"), dpi=200)
            plt.close()
        except Exception as exc:
            print(f"Warning: SHAP failed for {best_xgb_name}. Reason: {exc}")
else:
    print("SHAP is unavailable in this environment. Skipping SHAP analysis.")

## Early-warning threshold evaluation

In [ ]:
binary_rows = []
category_rows = []

for target_group in ["cost", "schedule"]:
    y_true = y_cost_test.to_numpy() if target_group == "cost" else y_schedule_test.to_numpy()
    actual_categories = assign_risk_category(y_true, *RISK_BAND_EDGES)

    for model_name, y_pred in predictions[target_group].items():
        binary_result = compute_high_risk_metrics(y_true, y_pred, threshold=HIGH_RISK_BINARY_THRESHOLD)
        binary_result["target_group"] = target_group
        binary_result["model_name"] = model_name
        binary_rows.append(binary_result)

        predicted_categories = assign_risk_category(y_pred, *RISK_BAND_EDGES)
        category_frame = pd.DataFrame(
            {
                "target_group": target_group,
                "model_name": model_name,
                "actual_risk_category": actual_categories,
                "predicted_risk_category": predicted_categories,
            }
        )
        summary = (
            category_frame.groupby(["predicted_risk_category"], as_index=False)
            .size()
            .rename(columns={"size": "n_projects"})
        )
        summary["share_projects"] = summary["n_projects"] / len(category_frame)
        summary["target_group"] = target_group
        summary["model_name"] = model_name
        category_rows.append(summary)

binary_eval_df = pd.DataFrame(binary_rows).sort_values(["target_group", "f1"], ascending=[True, False])
category_summary_df = pd.concat(category_rows, ignore_index=True).sort_values(["target_group", "model_name", "predicted_risk_category"])

binary_eval_df.to_csv(os.path.join(TABLES_DIR, "model_review_high_risk_binary_evaluation.csv"), index=False)
category_summary_df.to_csv(os.path.join(TABLES_DIR, "model_review_risk_category_summary.csv"), index=False)

print("Binary high-risk detection (threshold = 15%)")
display(binary_eval_df)
print("Risk-category distribution from regression outputs")
display(category_summary_df)

## Save outputs

In [ ]:
saved_tables = sorted(glob.glob(os.path.join(TABLES_DIR, "model_review_*.csv")))
saved_figures = sorted(glob.glob(os.path.join(FIGURES_DIR, "model_review_*.png")))

print("Saved tables:")
for path in saved_tables:
    print(" -", os.path.basename(path))

print("\nSaved figures:")
for path in saved_figures:
    print(" -", os.path.basename(path))

## Short final summary

In [ ]:
def summarize_best_model(metric_table, target_label):
    if metric_table.empty:
        print(f"No evaluated models were available for {target_label}.")
        return None
    best_row = metric_table.sort_values("rmse").iloc[0]
    print(f"Best {target_label} model: {best_row['model_name']}")
    print(f"RMSE = {best_row['rmse']:.4f}, MAE = {best_row['mae']:.4f}, MAPE = {best_row['mape']:.4f}, R2 = {best_row['r2']:.4f}")
    return best_row


best_cost = summarize_best_model(cost_metrics_df, "cost")
print()
best_schedule = summarize_best_model(schedule_metrics_df, "schedule")
print()

for target_label, table in [("cost", cost_metrics_df), ("schedule", schedule_metrics_df)]:
    linear_rows = table[table["model_name"].str.contains("linear", case=False, na=False)]
    tree_rows = table[table["model_name"].str.contains("forest|xgb|xgboost", case=False, na=False)]
    if linear_rows.empty or tree_rows.empty:
        print(f"Tree-vs-linear comparison for {target_label} could not be completed.")
        continue
    best_linear_rmse = linear_rows["rmse"].min()
    best_tree_rmse = tree_rows["rmse"].min()
    if best_tree_rmse < best_linear_rmse:
        print(f"For {target_label}, at least one tree model outperformed linear regression by RMSE.")
    else:
        print(f"For {target_label}, linear regression was not outperformed by the tree models in this review.")

## Feature Importance Analysis

In [ ]:
# This section appends a thesis-focused feature-importance review for tree-based models.

def infer_raw_feature_name(transformed_feature_name):
    if transformed_feature_name.startswith("num__"):
        return transformed_feature_name.split("num__", 1)[1]
    if transformed_feature_name.startswith("cat__"):
        cat_name = transformed_feature_name.split("cat__", 1)[1]
        for raw_col in X_test.columns:
            prefix = raw_col + "_"
            if cat_name == raw_col or cat_name.startswith(prefix):
                return raw_col
        return cat_name
    return transformed_feature_name


tree_feature_tables = []
tree_model_names = [name for name in loaded_models if infer_model_family(name) in {"random_forest", "xgboost"}]
print(f"Tree-based models found: {tree_model_names}")

for model_name in tree_model_names:
    model = loaded_models[model_name]
    target_group = infer_target_from_name(model_name)
    try:
        estimator = model.named_steps["model"] if hasattr(model, "named_steps") and "model" in model.named_steps else model
        if not hasattr(estimator, "feature_importances_"):
            print(f"Warning: {model_name} does not expose feature_importances_. Skipping.")
            continue

        transformed_names = get_transformed_feature_names(model, X_test.columns)
        raw_feature_names = [infer_raw_feature_name(name) for name in transformed_names]
        importance_df = pd.DataFrame(
            {
                "feature": raw_feature_names,
                "importance": estimator.feature_importances_,
            }
        )
        importance_df = importance_df.groupby("feature", as_index=False)["importance"].sum()
        importance_df["model_name"] = model_name
        importance_df["target_group"] = target_group
        importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
        tree_feature_tables.append(importance_df)

        output_table = os.path.join(TABLES_DIR, f"model_review_feature_importance_table_{model_name}.csv")
        importance_df.to_csv(output_table, index=False)
        print(f"Top 10 features for {model_name}")
        display(importance_df.head(10))

        top10_df = importance_df.head(10).sort_values("importance", ascending=True)
        plt.figure(figsize=(10, 6))
        sns.barplot(data=top10_df, x="importance", y="feature", hue="feature", legend=False, palette="deep")
        plt.title(f"Top 10 feature importances: {model_name}")
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, f"model_review_feature_importance_top10_{model_name}.png"), dpi=200)
        plt.close()
    except Exception as exc:
        print(f"Warning: feature-importance analysis failed for {model_name}. Reason: {exc}")

if tree_feature_tables:
    all_tree_importance_df = pd.concat(tree_feature_tables, ignore_index=True)
    all_tree_importance_df.to_csv(os.path.join(TABLES_DIR, "model_review_feature_importance_all_tree_models.csv"), index=False)
else:
    all_tree_importance_df = pd.DataFrame()
    print("No tree-based feature-importance tables were generated.")

if SHAP_AVAILABLE:
    for target_group in ["cost", "schedule"]:
        target_metrics = metrics_df[
            (metrics_df["target_group"] == target_group)
            & (metrics_df["model_name"].isin(tree_model_names))
        ].sort_values("rmse")
        if target_metrics.empty:
            print(f"No tree-based model available for SHAP analysis on target '{target_group}'.")
            continue

        best_tree_name = target_metrics.iloc[0]["model_name"]
        best_tree_model = loaded_models[best_tree_name]
        try:
            X_aligned, is_valid = validate_model_input(best_tree_name, best_tree_model, X_test)
            if not is_valid:
                print(f"Warning: SHAP skipped for {best_tree_name} because input validation failed.")
                continue

            if not hasattr(best_tree_model, "named_steps") or "preprocessor" not in best_tree_model.named_steps:
                print(f"Warning: SHAP skipped for {best_tree_name} because no preprocessor was found.")
                continue

            preprocessor = best_tree_model.named_steps["preprocessor"]
            estimator = best_tree_model.named_steps["model"]
            X_transformed = preprocessor.transform(X_aligned)
            transformed_names = preprocessor.get_feature_names_out()
            X_shap = X_transformed[:500] if X_transformed.shape[0] > 500 else X_transformed
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(X_shap)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_values, X_shap, feature_names=transformed_names, show=False)
            plt.title(f"SHAP summary: best tree model for {target_group} ({best_tree_name})")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_feature_importance_shap_{best_tree_name}.png"), dpi=200)
            plt.close()
        except Exception as exc:
            print(f"Warning: SHAP analysis failed for {best_tree_name}. Reason: {exc}")
else:
    print("SHAP is unavailable. SHAP summary plots were skipped.")

## Predicted vs Actual Diagnostic

In [ ]:
# This section checks whether predictions look unrealistically deterministic.
diagnostic_rows = []

for target_group in ["cost", "schedule"]:
    target_subset = prediction_frame[prediction_frame["target_group"] == target_group].copy()
    for model_name in target_subset["model_name"].unique():
        model_subset = target_subset[target_subset["model_name"] == model_name].copy()
        actual = model_subset["actual"].to_numpy()
        predicted = model_subset["predicted"].to_numpy()
        residuals = model_subset["residual"].to_numpy()
        safe_name = model_name.replace("/", "_")

        try:
            corr = float(np.corrcoef(actual, predicted)[0, 1]) if len(actual) > 1 else np.nan
            slope, intercept = np.polyfit(actual, predicted, 1)
            diagnostic_rows.append(
                {
                    "target_group": target_group,
                    "model_name": model_name,
                    "pred_actual_correlation": corr,
                    "pred_on_actual_slope": float(slope),
                    "pred_on_actual_intercept": float(intercept),
                    "mean_residual": float(np.mean(residuals)),
                    "residual_std": float(np.std(residuals, ddof=1)),
                }
            )

            plt.figure(figsize=(8, 6))
            plt.scatter(actual, predicted, alpha=0.65)
            xy_min = min(actual.min(), predicted.min())
            xy_max = max(actual.max(), predicted.max())
            plt.plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--", color="black")
            plt.title(f"Predicted vs actual diagnostic: {model_name} ({target_group})")
            plt.xlabel("Actual overrun (%)")
            plt.ylabel("Predicted overrun (%)")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_predicted_vs_actual_{target_group}_{safe_name}.png"), dpi=200)
            plt.close()

            plt.figure(figsize=(8, 6))
            sns.histplot(residuals, kde=True, bins=30)
            plt.title(f"Residual histogram: {model_name} ({target_group})")
            plt.xlabel("Residual")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_residual_histogram_{target_group}_{safe_name}.png"), dpi=200)
            plt.close()

            plt.figure(figsize=(8, 6))
            plt.scatter(predicted, residuals, alpha=0.65)
            plt.axhline(0.0, linestyle="--", color="black")
            plt.title(f"Residuals vs predicted: {model_name} ({target_group})")
            plt.xlabel("Predicted overrun (%)")
            plt.ylabel("Residual")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"model_review_residuals_vs_predicted_{target_group}_{safe_name}.png"), dpi=200)
            plt.close()
        except Exception as exc:
            print(f"Warning: predicted-vs-actual diagnostics failed for {model_name}. Reason: {exc}")

diagnostic_summary_df = pd.DataFrame(diagnostic_rows).sort_values(["target_group", "pred_actual_correlation"], ascending=[True, False])
diagnostic_summary_df.to_csv(os.path.join(TABLES_DIR, "model_review_predicted_actual_diagnostic_summary.csv"), index=False)
display(diagnostic_summary_df)

If the points lie extremely tightly on the 45-degree line, the simulation may be too deterministic. If there is visible but moderate dispersion around the diagonal, the simulation has realistic noise. A perfect line is suspicious. A moderate cloud around the diagonal is preferable for thesis realism.

## Thesis Results Tables and Figures

In [ ]:
# Create thesis-labelled tables and keep the formatting simple and transparent.

THESIS_DPI = 300

def pretty_model_name(model_name):
    lower = model_name.lower()
    if "linear" in lower:
        return "Linear Regression"
    if "random_forest" in lower or "forest" in lower:
        return "Random Forest"
    if "xgboost" in lower or "xgb" in lower:
        return "XGBoost"
    return model_name


def bootstrap_ci_row(y_true, y_pred, model_label, target_label, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    rmse_vals, mae_vals, r2_vals = [], [], []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        rmse_vals.append(np.sqrt(mean_squared_error(yt, yp)))
        mae_vals.append(mean_absolute_error(yt, yp))
        r2_vals.append(r2_score(yt, yp))
    return {
        "Target": target_label,
        "Model": model_label,
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "RMSE 95% CI": f"[{np.quantile(rmse_vals, 0.025):.4f}, {np.quantile(rmse_vals, 0.975):.4f}]",
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAE 95% CI": f"[{np.quantile(mae_vals, 0.025):.4f}, {np.quantile(mae_vals, 0.975):.4f}]",
        "R2": float(r2_score(y_true, y_pred)),
        "R2 95% CI": f"[{np.quantile(r2_vals, 0.025):.4f}, {np.quantile(r2_vals, 0.975):.4f}]",
    }


table_4_1 = cost_metrics_df.copy()
table_4_1["Model"] = table_4_1["model_name"].map(pretty_model_name)
table_4_1 = table_4_1[["Model", "rmse", "mae", "mape", "r2"]].rename(
    columns={"rmse": "RMSE", "mae": "MAE", "mape": "MAPE", "r2": "R2"}
)
table_4_1.to_csv(os.path.join(TABLES_DIR, "table_4_1_cost_model_performance.csv"), index=False)
print("Table 4.1 Predictive Performance of Regression Models for Cost Overrun Prediction")
display(table_4_1)

table_4_2 = schedule_metrics_df.copy()
table_4_2["Model"] = table_4_2["model_name"].map(pretty_model_name)
table_4_2 = table_4_2[["Model", "rmse", "mae", "mape", "r2"]].rename(
    columns={"rmse": "RMSE", "mae": "MAE", "mape": "MAPE", "r2": "R2"}
)
table_4_2.to_csv(os.path.join(TABLES_DIR, "table_4_2_schedule_model_performance.csv"), index=False)
print("Table 4.2 Predictive Performance of Regression Models for Schedule Overrun Prediction")
display(table_4_2)

cost_ew = binary_eval_df[binary_eval_df["target_group"] == "cost"].copy()
cost_ew["Model"] = cost_ew["model_name"].map(pretty_model_name)
table_4_3 = cost_ew[["Model", "precision", "recall", "f1", "tp", "fp", "tn", "fn"]].rename(
    columns={
        "precision": "Precision",
        "recall": "Recall",
        "f1": "F1-score",
        "tp": "True Positives",
        "fp": "False Positives",
        "tn": "True Negatives",
        "fn": "False Negatives",
    }
)
table_4_3.to_csv(os.path.join(TABLES_DIR, "table_4_3_cost_early_warning_detection.csv"), index=False)
print("Table 4.3 Early-Warning Detection Performance for Cost Overruns (15% Threshold)")
display(table_4_3)

schedule_ew = binary_eval_df[binary_eval_df["target_group"] == "schedule"].copy()
schedule_ew["Model"] = schedule_ew["model_name"].map(pretty_model_name)
table_4_4 = schedule_ew[["Model", "precision", "recall", "f1", "tp", "fp", "tn", "fn"]].rename(
    columns={
        "precision": "Precision",
        "recall": "Recall",
        "f1": "F1-score",
        "tp": "True Positives",
        "fp": "False Positives",
        "tn": "True Negatives",
        "fn": "False Negatives",
    }
)
table_4_4.to_csv(os.path.join(TABLES_DIR, "table_4_4_schedule_early_warning_detection.csv"), index=False)
print("Table 4.4 Early-Warning Detection Performance for Schedule Overruns (15% Threshold)")
display(table_4_4)

best_cost_row = cost_metrics_df.sort_values("rmse").iloc[0]
best_schedule_row = schedule_metrics_df.sort_values("rmse").iloc[0]
best_cost_pred = predictions["cost"][best_cost_row["model_name"]]
best_schedule_pred = predictions["schedule"][best_schedule_row["model_name"]]

table_4_5 = pd.DataFrame(
    [
        bootstrap_ci_row(y_cost_test.to_numpy(), best_cost_pred, pretty_model_name(best_cost_row["model_name"]), "Cost"),
        bootstrap_ci_row(y_schedule_test.to_numpy(), best_schedule_pred, pretty_model_name(best_schedule_row["model_name"]), "Schedule"),
    ]
)
table_4_5.to_csv(os.path.join(TABLES_DIR, "table_4_5_bootstrap_confidence_intervals.csv"), index=False)
print("Table 4.5 Bootstrap Confidence Intervals for Model Performance Metrics")
display(table_4_5)

In [ ]:
def save_thesis_figure(fig, base_name):
    fig.savefig(os.path.join(FIGURES_DIR, base_name + ".png"), dpi=THESIS_DPI, bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, base_name + ".pdf"), bbox_inches="tight")


def add_large_font_style(ax):
    ax.tick_params(labelsize=12)
    ax.title.set_fontsize(15)
    ax.xaxis.label.set_size(13)
    ax.yaxis.label.set_size(13)


# Figure 4.1
fig, ax = plt.subplots(figsize=(8, 6))
fig_4_1_df = table_4_1.copy()
sns.barplot(data=fig_4_1_df, x="Model", y="RMSE", hue="Model", legend=False, ax=ax, palette="deep")
ax.set_title("Figure 4.1 Comparison of RMSE Across Cost Overrun Prediction Models")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE")
add_large_font_style(ax)
save_thesis_figure(fig, "figure_4_1_cost_rmse_comparison")
plt.close(fig)

# Figure 4.2
fig, ax = plt.subplots(figsize=(8, 6))
fig_4_2_df = table_4_2.copy()
sns.barplot(data=fig_4_2_df, x="Model", y="RMSE", hue="Model", legend=False, ax=ax, palette="deep")
ax.set_title("Figure 4.2 Comparison of RMSE Across Schedule Overrun Prediction Models")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE")
add_large_font_style(ax)
save_thesis_figure(fig, "figure_4_2_schedule_rmse_comparison")
plt.close(fig)

# Figure 4.3
fig, ax = plt.subplots(figsize=(9, 6))
fig_4_3_df = pd.concat(
    [
        table_4_3[["Model", "F1-score"]].assign(Target="Cost"),
        table_4_4[["Model", "F1-score"]].assign(Target="Schedule"),
    ],
    ignore_index=True,
)
sns.barplot(data=fig_4_3_df, x="Model", y="F1-score", hue="Target", ax=ax, palette="deep")
ax.set_title("Figure 4.3 F1-Score Comparison for High-Risk Project Detection")
ax.set_xlabel("Model")
ax.set_ylabel("F1-score")
add_large_font_style(ax)
save_thesis_figure(fig, "figure_4_3_high_risk_f1_comparison")
plt.close(fig)

# Figures 4.4 and 4.5 using Random Forest model feature importances.
rf_importance_df = all_tree_importance_df[all_tree_importance_df["model_name"].str.contains("random_forest", case=False, na=False)].copy() if not all_tree_importance_df.empty else pd.DataFrame()
for target_group, figure_name, title_text in [
    ("cost", "figure_4_4_cost_random_forest_feature_importance", "Figure 4.4 Feature Importance Ranking for Cost Overrun Prediction (Random Forest Model)"),
    ("schedule", "figure_4_5_schedule_random_forest_feature_importance", "Figure 4.5 Feature Importance Ranking for Schedule Overrun Prediction (Random Forest Model)"),
]:
    subset = rf_importance_df[rf_importance_df["target_group"] == target_group].copy()
    if subset.empty:
        print(f"Warning: no Random Forest importance table found for {target_group}.")
        continue
    top_subset = subset.head(10).sort_values("importance", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.barplot(data=top_subset, x="importance", y="feature", hue="feature", legend=False, ax=ax, palette="deep")
    ax.set_title(title_text)
    ax.set_xlabel("Feature importance")
    ax.set_ylabel("Feature")
    add_large_font_style(ax)
    save_thesis_figure(fig, figure_name)
    plt.close(fig)

# Figures 4.6 and 4.7 are optional SHAP figures if SHAP runs.
if SHAP_AVAILABLE:
    for target_group, figure_name in [
        ("cost", "figure_4_6_shap_cost_overrun_summary"),
        ("schedule", "figure_4_7_shap_schedule_overrun_summary"),
    ]:
        tree_metrics = metrics_df[
            (metrics_df["target_group"] == target_group)
            & (metrics_df["model_name"].str.contains("forest|xgb|xgboost", case=False, na=False))
        ].sort_values("rmse")
        if tree_metrics.empty:
            continue
        best_tree_name = tree_metrics.iloc[0]["model_name"]
        best_tree_model = loaded_models[best_tree_name]
        try:
            X_aligned, is_valid = validate_model_input(best_tree_name, best_tree_model, X_test)
            if not is_valid or not hasattr(best_tree_model, "named_steps"):
                continue
            preprocessor = best_tree_model.named_steps["preprocessor"]
            estimator = best_tree_model.named_steps["model"]
            X_transformed = preprocessor.transform(X_aligned)
            X_shap = X_transformed[:500] if X_transformed.shape[0] > 500 else X_transformed
            feature_names = preprocessor.get_feature_names_out()
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(X_shap)
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False)
            plt.title(f"{figure_name.replace('_', ' ').title()}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, figure_name + ".png"), dpi=THESIS_DPI, bbox_inches="tight")
            plt.savefig(os.path.join(FIGURES_DIR, figure_name + ".pdf"), bbox_inches="tight")
            plt.close()
        except Exception as exc:
            print(f"Warning: thesis SHAP figure failed for {best_tree_name}. Reason: {exc}")

# Figures 4.8 to 4.11 for best-performing models.
for target_group, best_row, y_true, y_pred, scatter_name, scatter_title, residual_name, residual_title in [
    ("cost", best_cost_row, y_cost_test.to_numpy(), best_cost_pred, "figure_4_8_predicted_vs_actual_cost", "Figure 4.8 Predicted vs Actual Cost Overrun Values for the Best Performing Model", "figure_4_10_residual_distribution_cost", "Figure 4.10 Residual Distribution for Cost Overrun Prediction"),
    ("schedule", best_schedule_row, y_schedule_test.to_numpy(), best_schedule_pred, "figure_4_9_predicted_vs_actual_schedule", "Figure 4.9 Predicted vs Actual Schedule Overrun Values for the Best Performing Model", "figure_4_11_residual_distribution_schedule", "Figure 4.11 Residual Distribution for Schedule Overrun Prediction"),
]:
    residuals = y_true - y_pred
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(y_true, y_pred, alpha=0.65)
    xy_min = min(np.min(y_true), np.min(y_pred))
    xy_max = max(np.max(y_true), np.max(y_pred))
    ax.plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--", color="black")
    ax.set_title(scatter_title)
    ax.set_xlabel("Actual overrun")
    ax.set_ylabel("Predicted overrun")
    add_large_font_style(ax)
    save_thesis_figure(fig, scatter_name)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.histplot(residuals, kde=True, bins=30, ax=ax)
    ax.set_title(residual_title)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Frequency")
    add_large_font_style(ax)
    save_thesis_figure(fig, residual_name)
    plt.close(fig)

print("Thesis figures saved to outputs/figures as PNG and PDF.")

## Thesis Results Summary

This final block is intended for Chapter 4 style reporting: identify the best cost model, the best schedule model, summarize early-warning performance at the 15% threshold, and discuss the main predictors shown by Random Forest and optional SHAP outputs.